# KhushiDL - Heart Disease Detector

In [728]:
# Importing necessary modules
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import StandardScaler

## Data Preprocessing

In [729]:
# Loading the data
train_data = pd.read_csv('data/train.csv', header=None)
validation_data = pd.read_csv('data/validation.csv', header=None)
test_data = pd.read_csv('data/test.csv', header=None)

In [730]:
print("Shape: ", train_data.shape)
train_data.info()

Shape:  (644, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 644 entries, 0 to 643
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       644 non-null    float64
 1   1       644 non-null    float64
 2   2       644 non-null    float64
 3   3       644 non-null    float64
 4   4       644 non-null    float64
 5   5       644 non-null    float64
 6   6       644 non-null    float64
 7   7       644 non-null    float64
 8   8       644 non-null    float64
 9   9       644 non-null    float64
 10  10      644 non-null    float64
 11  11      644 non-null    float64
 12  12      644 non-null    float64
 13  13      644 non-null    float64
dtypes: float64(14)
memory usage: 70.6 KB


In [731]:
# seperating params
train_x = train_data.iloc[:, [x for x in range(13)]]
validation_x = validation_data.iloc[:, [x for x in range(13)]]
test_x = test_data.iloc[:, [x for x in range(13)]]

In [732]:
print("Train Shape: ", train_x.shape)
print("Validation Shape: ", validation_x.shape)
print("Test Shape: ", test_x.shape)

Train Shape:  (644, 13)
Validation Shape:  (138, 13)
Test Shape:  (138, 13)


In [733]:
# sperating labels
train_y = train_data.iloc[: , [13]].copy()
validation_y = validation_data.iloc[: , [13]].copy()
test_y = test_data.iloc[: , [13]].copy()

In [734]:
print("Train Shape: ", train_y.shape)
print("Validation Shape: ", validation_y.shape)
print("Test Shape: ", test_y.shape)

Train Shape:  (644, 1)
Validation Shape:  (138, 1)
Test Shape:  (138, 1)


In [735]:
test_y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   13      138 non-null    float64
dtypes: float64(1)
memory usage: 1.2 KB


In [736]:
# Using standar scaler
scaler = StandardScaler()

train_x = scaler.fit_transform(train_x)

validation_x = scaler.transform(validation_x)
test_x = scaler.transform(test_x)

In [737]:
# Clipping 1,2,3,4 into 1 single class (to improve model accuracy)
train_y = train_y.replace(2, 1)
train_y = train_y.replace(3, 1)
train_y = train_y.replace(4, 1)

validation_y = validation_y.replace(2, 1)
validation_y = validation_y.replace(3, 1)
validation_y = validation_y.replace(4, 1)

test_y = test_y.replace(2, 1)
test_y = test_y.replace(3, 1)
test_y = test_y.replace(4, 1)

In [738]:
# testing data
print(train_y.value_counts())
print(validation_y.value_counts())
print(test_y.value_counts())

13 
1.0    360
0.0    284
Name: count, dtype: int64
13 
1.0    84
0.0    54
Name: count, dtype: int64
13 
0.0    73
1.0    65
Name: count, dtype: int64


### Data Pipeline

In [739]:
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE


train_ds = (
    tf.data.Dataset.from_tensor_slices((train_x, train_y))
    .batch(BATCH)
    .shuffle(buffer_size=train_x.shape[0])
    .prefetch(AUTOTUNE)
)

validation_ds = (
    tf.data.Dataset.from_tensor_slices((validation_x, validation_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_x, test_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

## Model Architecture

In [740]:
# creating the actual model
model = tf.keras.Sequential([
    tf.keras.Input(shape=(13,)),
    
    # tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(50, activation='relu'),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_51"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_150 (Dense)               │ (None, 50)             │           700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_151 (Dense)               │ (None, 10)             │           510 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_152 (Dense)               │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,221 (4.77 KB)

 Trainable params: 1,221 (4.77 KB)

 Non-trainable params: 0 (0.00 B)

### Compiling and Training the Model

In [741]:
# Compiling the model
model.compile(
    optimizer='adam',
    # loss='sparse_categorical_crossentropy',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [742]:
# EarlyStopping callback
earlyStopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [743]:
# LR on Plateau callback
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=.8,
    patience=2
)

In [744]:
# training the model
model_history = model.fit(
    train_ds,
    validation_data = validation_ds,
    epochs=25,
    callbacks=[earlyStopper]
)

Epoch 1/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5901 - loss: 0.6556 - val_accuracy: 0.7029 - val_loss: 0.6174
Epoch 2/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7143 - loss: 0.6089 - val_accuracy: 0.7754 - val_loss: 0.5763
Epoch 3/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7717 - loss: 0.5750 - val_accuracy: 0.8188 - val_loss: 0.5413
Epoch 4/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8012 - loss: 0.5424 - val_accuracy: 0.8261 - val_loss: 0.5128
Epoch 5/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8059 - loss: 0.5149 - val_accuracy: 0.8261 - val_loss: 0.4898
Epoch 6/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8090 - loss: 0.4908 - val_accuracy: 0.8333 - val_loss: 0.4683
Epoch 7/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8121 - loss: 0.4671 - val_accuracy: 0.8406 - val_loss: 0.4488
Epoch 8/25
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8276 - loss: 0.4478 - val_accuracy: 0.8261 - val_loss

In [745]:
model.evaluate(test_ds)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8116 - loss: 0.4237


[0.42371830344200134, 0.8115941882133484]